In [ ]:
import pandas as pd
import glob
import matplotlib.pyplot as plt 
import seaborn as sns
import numpy as np
import matplotlib.cm as cm

# Read all CSV files from the specified directory
csv_files = glob.glob("data/txs/august/*_parsed.csv")
txs = pd.concat([pd.read_csv(f) for f in csv_files], ignore_index=True)
auctions = pd.read_csv("data/auctions/timeboost_auctions.csv")
binance_btcusd = pd.read_csv("data/prices/BTCUSDT-1s-2025-08.csv")
binance_ethusd = pd.read_csv("data/prices/ETHUSDT-1s-2025-08.csv")
binance_arbusd = pd.read_csv("data/prices/ARBUSDT-1s-2025-08.csv")

if 'merged' in txs.columns:
    txs = txs.drop(columns=['merged', 'auction_round', 'winner_address','winner_name','top_bid_eth','paid_bid_eth', 'express_lane_controller_address'])
    
# 1. Ensure proper datetime types and remove tz
txs["block_time"] = pd.to_datetime(txs["block_time"], utc=True).dt.tz_convert(None)
auctions["round_start_time"] = pd.to_datetime(auctions["round_start_time"], utc=True).dt.tz_convert(None)
auctions["round_end_time"]   = pd.to_datetime(auctions["round_end_time"], utc=True).dt.tz_convert(None)

# 2. Sort both frames
txs = txs.sort_values("block_time")
auctions = auctions.sort_values("round_start_time")


In [ ]:
# 3. Asof merge (assign nearest prior start)
txs = pd.merge_asof(
    txs,
    auctions[["auction_round", "round_start_time", "round_end_time"]],
    left_on="block_time",
    right_on="round_start_time",
    direction="backward"
)

# 4. REMOVE faulty assignments: keep only if block_time < round_end_time
txs = txs[txs["block_time"] < txs["round_end_time"]].copy()

# 5. Compute relative time
txs["time_since_round_start"] = (
    txs["block_time"] - txs["round_start_time"]
).dt.total_seconds()


In [ ]:
# Merge with auctions on auction_round to get more auction details
txs = txs.merge(
    auctions.drop(columns=["round_start_time", "round_end_time"]),
    on="auction_round",
    how="left"
)

In [ ]:
plt.figure(figsize=(12, 6))

# Plot stacked histogram
bin_count = len(txs["time_since_round_start"].unique())
ax = sns.histplot(
    data=txs,
    x="time_since_round_start",
    hue="timeboosted",
    bins=bin_count,
    stat="count",
    multiple="stack"
)

total_timeboosted = txs[txs["timeboosted"] == True].shape[0]
avg_timeboosted = (txs[txs["timeboosted"] == True].shape[0] / txs.shape[0]) * 100
print(f"Overall, {avg_timeboosted:.2f}% of CEX/DEX transactions are timeboosted ({total_timeboosted} out of {txs.shape[0]})")


plt.title("Distribution of CEX/DEX Transaction Times Within Auction Rounds (August 2025)")
plt.xlabel("Time Since Round Start (seconds)")
plt.ylabel("Number of Transactions")
plt.xlim(0, max(txs["time_since_round_start"]))

# ---------------------------------------------------------
# Annotate percentage labels on stacked bar sections
# ---------------------------------------------------------

# Get total counts in each bin
patches = ax.patches
# There are N bins × H hues stacked -> reshape
num_bins = bin_count
num_hues = txs["timeboosted"].nunique()

# # Convert patches → array: shape = (num_bins, num_hues)
# heights = np.array([p.get_height() for p in patches]).reshape(num_bins, num_hues)

# # Total per bin
# totals = heights.sum(axis=1)

# # Loop through bins and levels
# for i in range(num_bins):
#     cum_height = 0
#     for j in range(num_hues):
#         h = heights[i, j]
#         if h > 0:
#             pct = h / totals[i] * 100
#             # Center label vertically inside this stacked segment
#             ax.text(
#                 patches[i * num_hues + j].get_x() + patches[i * num_hues + j].get_width() / 2,
#                 cum_height + h / 2,
#                 f"{pct:.1f}%",
#                 ha="center",
#                 va="center",
#                 fontsize=7,
#                 color="white" if pct > 10 else "black"
#             )
#             cum_height += h

plt.tight_layout()
plt.show()


In [ ]:
# Print the distribution of txs to_address of all users
txs_to_address_counts = txs["tx_to_address"].value_counts()
print("Distribution of transactions to addresses:")
print(txs_to_address_counts)

In [ ]:
### --------------------------------------------------------------------
### CONFIG: Define MEV searcher bots
### --------------------------------------------------------------------
SEARCHERS = {
    "Wintermute": [
        '0xcb43d843f6cadf4f4844f3f57032468aadd9b95c',
        '0x27920e8039d2b6e93e36f5d5f53b998e2e631a70'
    ],
    "Selini": [
        '0xee2e7bbb67676292af2e31dffd1fea2276d6c7ba'
    ]
}


In [ ]:
### --------------------------------------------------------------------
### UTILITY: Weighted volume-stacked histogram
### --------------------------------------------------------------------
def plot_volume_histogram(df, hue_col, x_col, weight_col, bins):
    groups = df[hue_col].unique()
    colors = cm.get_cmap('tab10').colors

    # Weighted hist per group
    hist_data = {}
    for g in groups:
        subset = df[df[hue_col] == g]
        hist, bin_edges = np.histogram(
            subset[x_col],
            bins=bins,
            weights=subset[weight_col]
        )
        hist_data[g] = hist

    # Plot stacked bars manually
    plt.figure(dpi=300)
    bottom = np.zeros(len(bin_edges) - 1)

    for g, color in zip(groups, colors):
        plt.bar(
            bin_edges[:-1],
            hist_data[g],
            width=np.diff(bin_edges),
            bottom=bottom,
            color=color,
            edgecolor="black",
            label=g,
            align="edge"
        )
        bottom += hist_data[g]

    plt.xlabel("Time Since Round Start (seconds)")
    plt.ylabel("USD Volume")
    plt.title("CEX-DEX USD Volume Within Auction Rounds")
    plt.xlim(0, df[x_col].max())
    # Make ticks and labels more readable
    # plt.tick_params(fontsize=14)
    plt.legend()
    plt.tight_layout()
    plt.show()


### --------------------------------------------------------------------
### ANALYSIS FUNCTION
### --------------------------------------------------------------------
def analyze_searcher(name, bots, txs, auctions):
    """
    Run full count + volume analysis for a single MEV searcher.
    Produces stacked histograms and prints summary stats.
    """

    # Filter txs involving the searcher's bots
    txs_bot = txs[txs["tx_to_address"].isin(bots)]

    # Split by win status
    won = txs_bot[(txs_bot["winner_name"] == name) & (txs_bot["timeboosted"] == True)]
    not_won = txs_bot[txs_bot["winner_name"] != name]

    # Plot dataframe
    won_plot = won.copy()
    won_plot["group"] = f"{name} Won"

    not_won_plot = not_won.copy()
    not_won_plot["group"] = f"Non-{name} Won"

    df = pd.concat([won_plot, not_won_plot], ignore_index=True)

    # Number of bins
    bin_count = len(df["time_since_round_start"].unique())
    if bin_count < 10:
        bin_count = 10  # safety fallback

    ### ----------------------------------------------------------------
    ### PLOT 1: Count histogram (stacked)
    ### ----------------------------------------------------------------
    plt.figure(figsize=(12, 6))
    sns.histplot(
        data=df,
        x="time_since_round_start",
        bins=bin_count,
        hue="group",
        multiple="stack",
        stat="count",
        palette=cm.get_cmap('tab10').colors,
        edgecolor="black",
    )
    plt.title(f"{name} — CEX/DEX Transaction Counts within Auction Rounds")
    plt.xlabel("Time Since Round Start (seconds)")
    plt.ylabel("Number of Transactions")
    plt.xlim(0, df["time_since_round_start"].max())
    plt.show()

    ### ----------------------------------------------------------------
    ### PLOT 2: USD Volume histogram (stacked, weighted)
    ### ----------------------------------------------------------------
    plot_volume_histogram(
        df=df,
        hue_col="group",
        x_col="time_since_round_start",
        weight_col="amount_usd",
        bins=bin_count
    )

    ### ----------------------------------------------------------------
    ### METRICS
    ### ----------------------------------------------------------------
    august_auctions = auctions[auctions["round_start_time"].dt.month == 8]

    won_auctions = august_auctions[august_auctions["winner_name"] == name]["auction_round"].nunique()
    total_auctions = august_auctions["auction_round"].nunique()

    win_share = won_auctions / total_auctions * 100 if total_auctions > 0 else 0

    won_tx_share = len(won) / len(txs_bot) * 100 if len(txs_bot) > 0 else 0
    
    won_tx_volume_share = won["amount_usd"].sum() / txs_bot["amount_usd"].sum() * 100 if txs_bot["amount_usd"].sum() > 0 else 0

    print(f"{name} won {win_share:.2f}% of auctions in August "
          f"({won_auctions}/{total_auctions}).")

    print(f"{name} has {won_tx_share:.2f}% of their CEX/DEX trades timeboosted when they win.")

    print(f"{name} has {won_tx_volume_share:.2f}% of their CEX/DEX trade volume timeboosted when they win.")
    print("\n" + "-"*80 + "\n")



### --------------------------------------------------------------------
### RUN ANALYSIS FOR EACH SEARCHER
### --------------------------------------------------------------------
for name, bots in SEARCHERS.items():
    analyze_searcher(name, bots, txs, auctions)


In [ ]:
txs["block_time"] = pd.to_datetime(txs["block_time"], utc=True)
txs = txs.sort_values("block_time").reset_index(drop=True)

wm_txs = txs[txs["tx_to_address"].isin(SEARCHERS["Wintermute"])]
sel_txs = txs[txs["tx_to_address"].isin(SEARCHERS["Selini"])]

In [ ]:
cols = [
    "open_time_us", "open", "high", "low", "close", "volume",
    "close_time_us", "quote_vol", "trades", "taker_base", "taker_quote", "ignore"
]

binance_arbusd.columns = cols
binance_ethusd.columns = cols
binance_btcusd.columns = cols

horizons = [0, 5, 10, 300, 600, 1800]

def prepare_pricefeed(df, token):
    df = df.copy()
    df["timestamp"] = pd.to_datetime(df["open_time_us"] / 1e6, unit="s", utc=True)
    df["midprice"] = (df["high"] + df["low"]) / 2
    df = df[["timestamp", "midprice"]].sort_values("timestamp").reset_index(drop=True)
    df = df.rename(columns={"midprice": f"{token}_mid"})
    return df

binance_arbusd  = prepare_pricefeed(binance_arbusd, "ARB")
binance_ethusd  = prepare_pricefeed(binance_ethusd, "ETH")
binance_btcusd  = prepare_pricefeed(binance_btcusd, "BTC")

In [ ]:
def merge_markouts(sample, pricefeed, token, horizons=horizons):
    out = sample.copy()

    # PRICEFEED must not have duplicate timestamps or unnamed junk
    pf = pricefeed[["timestamp", f"{token}_mid"]].copy().sort_values("timestamp")

    for h in horizons:
        mark_col = f"{token}_mark_t{h}"
        price_col = f"{token}_mid_t{h}"

        # target timestamp for markout
        out[mark_col] = out["block_time"] + pd.to_timedelta(h, unit="s")

        # IMPORTANT: drop leftover timestamps from previous merges
        out = out.drop(columns=["timestamp"], errors="ignore")

        merged = pd.merge_asof(
            out.sort_values(mark_col),
            pf,
            left_on=mark_col,
            right_on="timestamp",
            direction="backward"
        )

        # Rename the merged midprice
        merged = merged.rename(columns={f"{token}_mid": price_col})

        # Drop redundant timestamp column from pricefeed
        merged = merged.drop(columns=["timestamp"], errors="ignore")

        out = merged  # continue loop

    return out

sample_txs = sel_txs.sample(100000)

sample_txs = merge_markouts(sample_txs, binance_arbusd, "ARB")
sample_txs = merge_markouts(sample_txs, binance_ethusd, "ETH")
sample_txs = merge_markouts(sample_txs, binance_btcusd, "BTC")

In [ ]:
stablecoins = {"USDC", "USD₮0"}

def get_price(token_symbol, row, h):
    """
    Returns the USD price for a token at horizon h
    """
    tok = token_symbol.replace("W", "")  # WETH→ETH, WBTC→BTC

    # Stablecoins: fixed $1
    if tok in stablecoins:
        return 1.0

    col = f"{tok}_mid_t{h}"
    return row[col] if col in row and pd.notna(row[col]) else None


def compute_pnl_row(row, h):
    buy_tok  = row["bought_token_symbol"]
    sell_tok = row["sold_token_symbol"]

    buy_price  = get_price(buy_tok, row, h)
    sell_price = get_price(sell_tok, row, h)

    # Fees in USD
    eth_fee_usd = row["tx_fee_eth"] * row["ETH_mid_t0"]

    # If feed missing → cannot compute PNL
    if buy_price is None or sell_price is None:
        return None

    pnl = (
        row["bought_token_amount"] * buy_price - 
        row["sold_token_amount"] * sell_price - 
        eth_fee_usd
    )
    return pnl


# Compute PNL for each horizon
for h in horizons:
    sample_txs[f"pnl_t{h}"] = sample_txs.apply(lambda row: compute_pnl_row(row, h), axis=1)


In [ ]:
daily_pnl = {}
sample_txs["date"] = sample_txs["block_time"].dt.date

for h in horizons:
    col = f"pnl_t{h}"

    # keep only txs with positive pnl at this horizon
    df_pos = sample_txs[sample_txs[col] > 0]

    df = (
        df_pos.groupby(["date", "timeboosted"])[col]
        .sum()
        .reset_index()
        .rename(columns={col: "pnl"})
    )

    daily_pnl[h] = df


In [ ]:
# Print aggregate stats for each markout horizon
for h in horizons:
    df = daily_pnl[h]

    # Total PNL timeboosted vs not
    total_timeboosted = df[df["timeboosted"] == True]["pnl"].sum()
    total_not_timeboosted = df[df["timeboosted"] == False]["pnl"].sum()

    overall_total = total_timeboosted + total_not_timeboosted
    pct_timeboosted = (total_timeboosted / overall_total * 100) if overall_total > 0 else 0

    print(f"Horizon {h} seconds:")
    print(f"  Total PNL Timeboosted: ${total_timeboosted:,.2f}")
    print(f"  Total PNL Not Timeboosted: ${total_not_timeboosted:,.2f}")
    print(f"  Percentage Timeboosted: {pct_timeboosted:.2f}%")
    print("\n" + "-"*60 + "\n")

In [ ]:
# Two rows, three columns
fig, axes = plt.subplots(2, 3, figsize=(18, 8), sharex=False)
axes = axes.flatten()  # flatten to iterate easily

for i, h in enumerate(horizons):
    ax = axes[i]
    df = daily_pnl[h]

    # Split into boosted/non-boosted
    boosted    = df[df["timeboosted"] == True]
    nonboosted = df[df["timeboosted"] == False]

    ax.plot(boosted["date"], boosted["pnl"], marker="o", label="Timeboosted", color="blue")
    ax.plot(nonboosted["date"], nonboosted["pnl"], marker="o", label="Non-Timeboosted", color="orange")

    ax.set_title(f"{h}s Horizon")
    # Tweak x-axis for better date formatting
    ax.xaxis.set_major_locator(plt.MaxNLocator(6))
    ax.set_xlabel("Date")
    ax.set_ylabel("PNL (USD)")
    ax.grid(True)
    ax.legend()

# Hide any unused subplot cells (if horizons < 6)
for j in range(len(horizons), 6):
    axes[j].axis("off")

plt.tight_layout()
plt.show()


### Auctions

In [ ]:
auctions['bid_diff'] = auctions['top_bid_eth'] - auctions['paid_bid_eth']
auctions['bid_diff_percent'] = auctions['bid_diff'] / auctions['top_bid_eth'] * 100
auctions['bid_ratio'] = auctions['paid_bid_eth'] / auctions['top_bid_eth'] * 100

auctions["round_start_time"] = pd.to_datetime(auctions["round_start_time"], utc=True).dt.tz_convert(None)
auctions["round_end_time"]   = pd.to_datetime(auctions["round_end_time"], utc=True).dt.tz_convert(None)

auctions = auctions.sort_values("round_start_time")

In [ ]:
# Plot the distribution of bid differences only 99th percentile
auctions_filtered = auctions[auctions['bid_diff'] <= auctions['bid_diff'].quantile(0.99)]
plt.figure(figsize=(10, 6))
sns.histplot(auctions_filtered['bid_diff'], bins=50, kde=True)
plt.title("Distribution of Bid Differences (Top Bid - Paid Bid)")
plt.xlabel("Bid Difference (ETH)")
plt.ylabel("Number of Auctions")
plt.show()

mean_bid_diff = auctions['bid_diff'].mean()
print(f"Mean Bid Difference (ETH): {mean_bid_diff}")
mean_bid_diff_percent = auctions['bid_diff_percent'].mean()
print(f"Mean Bid Difference (%): {mean_bid_diff_percent}")

plt.figure(figsize=(10, 6))
sns.histplot(auctions['bid_diff_percent'], bins=50, kde=True)
plt.title("Distribution of Bid Differences % (Top Bid - Paid Bid)")
plt.xlabel("Bid Difference (%)")
plt.ylabel("Number of Auctions")
plt.show()

In [ ]:
# Group auctions by date and calculate average bid difference per day
auctions['date'] = auctions['round_start_time'].dt.date
daily_avg_bid_diff = auctions.groupby('date')['bid_diff'].mean().reset_index()
daily_avg_bid_diff_percent = auctions.groupby('date')['bid_diff_percent'].mean().reset_index()
daily_avg_bid_ratio = auctions.groupby('date')['bid_ratio'].mean().reset_index()

# Group by winner_name and date to calculate average bid difference per winner per day
winner_daily_avg_bid_diff = auctions.groupby(['winner_name', 'date'])['bid_diff'].mean().reset_index()
winner_daily_avg_bid_diff_percent = auctions.groupby(['winner_name', 'date'])['bid_diff_percent'].mean().reset_index()
winner_daily_avg_bid_ratio = auctions.groupby(['winner_name', 'date'])['bid_ratio'].mean().reset_index()

In [ ]:
plt.figure(figsize=(12, 6))
sns.lineplot(data=daily_avg_bid_diff, x='date', y='bid_diff', marker='o')
plt.title("Average Daily Bid Difference Over Time")
plt.xlabel("Date")
plt.ylabel("Average Bid Difference (ETH)")
plt.show()

plt.figure(figsize=(12, 6))
sns.lineplot(data=daily_avg_bid_diff_percent, x='date', y='bid_diff_percent', marker='o')
plt.title("Average Daily Bid Difference % Over Time")
plt.xlabel("Date")
plt.ylabel("Average Bid Difference (%)")
plt.show()

plt.figure(figsize=(12, 6))
sns.lineplot(data=daily_avg_bid_ratio, x='date', y='bid_ratio', marker='o')
plt.title("Average Daily Bid Ratio Over Time")
plt.xlabel("Date")
plt.ylabel("Average Bid Ratio (%)")
# Add gridlines for better readability
plt.grid(True)
plt.show()


In [ ]:
# Plot average bid difference per winner over time
plt.figure(figsize=(12, 6))
sns.lineplot(data=winner_daily_avg_bid_diff, x='date', y='bid_diff', hue='winner_name', marker='o')
plt.title("Average Daily Bid Difference per Winner Over Time")
plt.xlabel("Date")
plt.ylabel("Average Bid Difference (ETH)")
plt.show()

#  Plot average bid difference % per winner over time
plt.figure(figsize=(12, 6))
sns.lineplot(data=winner_daily_avg_bid_diff_percent, x='date', y='bid_diff_percent', hue='winner_name', marker='o')
plt.title("Average Daily Bid Difference % per Winner Over Time")
plt.xlabel("Date")
plt.ylabel("Average Bid Difference (%)")
plt.show()

#  Plot average bid ratio per winner over time
plt.figure(figsize=(12, 6))
sns.lineplot(data=winner_daily_avg_bid_ratio, x='date', y='bid_ratio', hue='winner_name', marker='o')
plt.title("Average Daily Bid Ratio per Winner Over Time")
plt.xlabel("Date")
plt.ylabel("Average Bid Ratio (%)")
plt.show()

In [ ]:
# Per day, calculate win share by winner_name
auctions['date'] = auctions['round_start_time'].dt.date
daily_wins = auctions.groupby(['date', 'winner_name']).size().reset_index(name='win_count')
total_daily_wins = auctions.groupby('date').size().reset_index(name='total_wins')
daily_win_share = daily_wins.merge(total_daily_wins, on='date') 
daily_win_share['win_share'] = daily_win_share['win_count'] / daily_win_share['total_wins']


In [ ]:
import matplotlib.pyplot as plt

# Pivot so each winner is its own column
pivot = daily_win_share.pivot(index="date", columns="winner_name", values="win_share")

plt.figure(figsize=(14, 7))

bottom = None
for winner in pivot.columns:
    plt.bar(
        pivot.index,
        pivot[winner],
        bottom=bottom,
        label=winner
    )
    if bottom is None:
        bottom = pivot[winner]
    else:
        bottom = bottom + pivot[winner]

plt.title("Daily Win Share by Winner Name (Stacked Bar)")
plt.xlabel("Date")
plt.ylabel("Win Share")
plt.legend(title="Winner Name")
plt.xticks(rotation=45)
plt.tight_layout()
# Add gridlines for better readability
plt.grid(axis='y', linestyle='--')
# Limit x-axis to min and max dates in the data
plt.xlim(pivot.index.min(), pivot.index.max())
plt.show()


In [ ]:
auctions['biweekly'] = auctions['round_start_time'].dt.to_period('14D')

biweekly_wins = (
    auctions.groupby(['biweekly', 'winner_name'])
    .size()
    .reset_index(name='win_count')
)

total_biweekly_wins = (
    auctions.groupby('biweekly')
    .size()
    .reset_index(name='total_wins')
)

biweekly_win_share = biweekly_wins.merge(total_biweekly_wins, on='biweekly')
biweekly_win_share['win_share'] = (
    biweekly_win_share['win_count'] / biweekly_win_share['total_wins']
)
pivot_bw = biweekly_win_share.pivot(
    index='biweekly',
    columns='winner_name',
    values='win_share'
)

# Convert PeriodIndex → timestamp so matplotlib can plot it
pivot_bw.index = pivot_bw.index.to_timestamp()


In [ ]:
plt.figure(figsize=(14, 7))

plt.stackplot(
    pivot_bw.index,
    pivot_bw.T.values,   # each row = a winner
    labels=pivot_bw.columns,
    alpha=0.7
)

plt.title("Bi-Weekly Win Share by Winner (Stacked Area)")
plt.xlabel("Bi-Weekly Period")
plt.ylabel("Win Share")
plt.legend(title="Winner Name", loc="upper left")
plt.tight_layout()
# Add gridlines for better readability
plt.grid(axis='y', linestyle='--', color='white')
# Limit x-axis to min and max dates in the data
plt.xlim(pivot_bw.index.min(), pivot_bw.index.max())
# Limit y-axis to [0, 1]
plt.ylim(0, 1)
plt.show()